In [ ]:
import json

import numpy as np
import polars as pl

In [ ]:
# !hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir  ../data/vk-lsvd/raw

In [ ]:
# !hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/ur0.01_ir0.01/*" --local-dir ../data/vk-lsvd/raw

Partitioning Subsamples into Base, Gap, Validation, and Test Parts

A column named original_order is added to preserve the order within each part

In [ ]:
DATA_DIR = "../data"

subsample_name = "ur0.01_ir0.01"
NUM_PARTS = 10

DATASET_PATH = f"{DATA_DIR}/vk-lsvd/raw"
OUTPUT_DIR = f"{DATA_DIR}/vk-lsvd"

metadata_files = ["metadata/users_metadata.parquet", "metadata/items_metadata.parquet", "metadata/item_embeddings.npz"]


all_interactions_files = [f"subsamples/{subsample_name}/train/week_{i:02}.parquet" for i in range(25)]

len(all_interactions_files)

In [ ]:
def get_parquet_interactions(data_files, positive_event_timespent):
    data_interactions = pl.concat([pl.scan_parquet(f"{DATASET_PATH}/{file}") for file in data_files])
    data_interactions = data_interactions.collect()
    data_interactions = data_interactions.with_row_index("original_order")
    data_interactions = data_interactions.filter(pl.col("timespent") > positive_event_timespent)
    return data_interactions

In [ ]:
POSITIVE_EVENT_TIMESPENT = 15
all_data_interactions = get_parquet_interactions(all_interactions_files, POSITIVE_EVENT_TIMESPENT)

Loading and Filtering Embeddings

In [ ]:
all_data_users = all_data_interactions.select("user_id").unique()
all_data_items = all_data_interactions.select("item_id").unique()

item_ids = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")["item_id"]
item_embeddings = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")["embedding"]

mask = np.isin(item_ids, all_data_items.to_numpy())
item_ids = item_ids[mask]
item_embeddings = item_embeddings[mask]

items_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/items_metadata.parquet")

items_metadata = items_metadata.join(all_data_items, on="item_id")
items_metadata = items_metadata.join(pl.DataFrame({"item_id": item_ids, "embedding": item_embeddings}), on="item_id")

Remapping Item IDs

In [ ]:
all_data_items = all_data_interactions.select("item_id").unique()
all_data_users = all_data_interactions.select("user_id").unique()

unique_items_sorted = all_data_items.sort("item_id").with_row_index("new_item_id")
global_item_mapping = dict(zip(unique_items_sorted["item_id"], unique_items_sorted["new_item_id"]))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

In [ ]:
def remap_interactions(df, mapping):
    return df.with_columns(pl.col("item_id").map_elements(lambda x: mapping.get(x, None), return_dtype=pl.UInt32))


all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping)
items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping)

In [ ]:
all_data_interactions_remapped_sorted = all_data_interactions_remapped.sort("original_order")
all_data_interactions_remapped_sorted = all_data_interactions_remapped_sorted.with_columns(
    pl.int_range(0, pl.len()).alias("row_nr")
)

N = all_data_interactions_remapped_sorted.height
print(f"Total interactions: {N}")

base_size = N // NUM_PARTS
num_larger = N % NUM_PARTS
larger_group_size = base_size + 1

print(f"Base part size: {base_size}")
print(f"Number of larger parts: {num_larger}")
print(f"Size of larger parts: {larger_group_size}")

all_data_interactions_with_groups = all_data_interactions_remapped_sorted.with_columns(
    pl.when(pl.col("row_nr") < num_larger * larger_group_size)
    .then(pl.col("row_nr") // larger_group_size)
    .otherwise(num_larger + (pl.col("row_nr") - num_larger * larger_group_size) // base_size)
    .alias("part")
).drop("row_nr")

In [ ]:
all_data_interactions_with_groups.head

In [ ]:
parts_distribution = (
    all_data_interactions_with_groups.group_by("part")
    .agg(
        pl.count().alias("count"),
        pl.col("original_order").min().alias("min_order"),
        pl.col("original_order").max().alias("max_order"),
    )
    .sort("part")
)

print("Distribution by part:")
print(parts_distribution)

print(f"Minimum part: {parts_distribution['part'].min()}")
print(f"Maximum part: {parts_distribution['part'].max()}")
print(f"Total number of events from all parts: {parts_distribution['count'].sum()}")
print(f"Total number of events overall: {N}")

Saving

In [ ]:
mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, "w") as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Mapping saved: {mapping_output_path}")

In [ ]:
def write_parquet(output_dir, data, file_name):
    print(f"Shape: {data.shape}")
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"File saved: {file_name}")


write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, all_data_interactions_with_groups, "all_data_interactions_with_groups")